<a href="https://colab.research.google.com/github/komalshelar16/Natural-Language-Processing/blob/main/NLP_Exp_8_TF_IDF_%26_N_Gram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
documents = [
    "Natural language processing enables computers to understand human language.",
    "Machine learning is used in natural language processing.",
    "TF IDF is a technique used in text mining and natural language processing.",
    "Text mining and machine learning are important fields in data science."
]
print(documents)

['Natural language processing enables computers to understand human language.', 'Machine learning is used in natural language processing.', 'TF IDF is a technique used in text mining and natural language processing.', 'Text mining and machine learning are important fields in data science.']


In [6]:
nltk.download('punkt_tab') # Download the missing 'punkt_tab' resource

stop_words = set(stopwords.words('english'))

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha()]  # remove punctuation
    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords
    return tokens

processed_docs = [preprocess(doc) for doc in documents]

print("Processed Documents:\n", processed_docs)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Processed Documents:
 [['natural', 'language', 'processing', 'enables', 'computers', 'understand', 'human', 'language'], ['machine', 'learning', 'used', 'natural', 'language', 'processing'], ['tf', 'idf', 'technique', 'used', 'text', 'mining', 'natural', 'language', 'processing'], ['text', 'mining', 'machine', 'learning', 'important', 'fields', 'data', 'science']]


In [7]:
vocab = sorted(set(word for doc in processed_docs for word in doc))
vocab_index = {word: i for i, word in enumerate(vocab)}

print("\nVocabulary:\n", vocab)


Vocabulary:
 ['computers', 'data', 'enables', 'fields', 'human', 'idf', 'important', 'language', 'learning', 'machine', 'mining', 'natural', 'processing', 'science', 'technique', 'text', 'tf', 'understand', 'used']


In [8]:
N = len(processed_docs)   # Number of documents
V = len(vocab)            # Vocabulary size

tf_matrix = np.zeros((N, V))

for doc_idx, doc in enumerate(processed_docs):
    word_counts = Counter(doc)
    total_words = len(doc)

    for word, count in word_counts.items():
        tf_matrix[doc_idx, vocab_index[word]] = count / total_words

print("\nTF Matrix:\n", tf_matrix)


TF Matrix:
 [[0.125      0.         0.125      0.         0.125      0.
  0.         0.25       0.         0.         0.         0.125
  0.125      0.         0.         0.         0.         0.125
  0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.16666667 0.16666667 0.16666667 0.         0.16666667
  0.16666667 0.         0.         0.         0.         0.
  0.16666667]
 [0.         0.         0.         0.         0.         0.11111111
  0.         0.11111111 0.         0.         0.11111111 0.11111111
  0.11111111 0.         0.11111111 0.11111111 0.11111111 0.
  0.11111111]
 [0.         0.125      0.         0.125      0.         0.
  0.125      0.         0.125      0.125      0.125      0.
  0.         0.125      0.         0.125      0.         0.
  0.        ]]


In [9]:
idf_vector = np.zeros(V)

for word, idx in vocab_index.items():
    doc_count = sum(1 for doc in processed_docs if word in doc)
    idf_vector[idx] = np.log(N / doc_count)

print("\nIDF Vector:\n", idf_vector)


IDF Vector:
 [1.38629436 1.38629436 1.38629436 1.38629436 1.38629436 1.38629436
 1.38629436 0.28768207 0.69314718 0.69314718 0.69314718 0.28768207
 0.28768207 1.38629436 1.38629436 0.69314718 1.38629436 1.38629436
 0.69314718]


In [10]:
tfidf_matrix = tf_matrix * idf_vector

print("\nTF-IDF Matrix:\n", tfidf_matrix)


TF-IDF Matrix:
 [[0.1732868  0.         0.1732868  0.         0.1732868  0.
  0.         0.07192052 0.         0.         0.         0.03596026
  0.03596026 0.         0.         0.         0.         0.1732868
  0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.04794701 0.11552453 0.11552453 0.         0.04794701
  0.04794701 0.         0.         0.         0.         0.
  0.11552453]
 [0.         0.         0.         0.         0.         0.15403271
  0.         0.03196467 0.         0.         0.07701635 0.03196467
  0.03196467 0.         0.15403271 0.07701635 0.15403271 0.
  0.07701635]
 [0.         0.1732868  0.         0.1732868  0.         0.
  0.1732868  0.         0.0866434  0.0866434  0.0866434  0.
  0.         0.1732868  0.         0.0866434  0.         0.
  0.        ]]


In [11]:
import random
from collections import defaultdict

class NGramModel:
    def __init__(self, n):
        self.n = n
        self.ngrams = defaultdict(list)

    def train(self, text):
        tokens = text.split()

        for i in range(len(tokens) - self.n + 1):
            key = tuple(tokens[i:i + self.n - 1])
            next_word = tokens[i + self.n - 1]
            self.ngrams[key].append(next_word)

    def generate(self, start_words, length=20):
        current = tuple(start_words)
        output = list(start_words)

        for _ in range(length):
            if current in self.ngrams:
                next_word = random.choice(self.ngrams[current])
                output.append(next_word)
                current = tuple(output[-(self.n - 1):])
            else:
                break

        return " ".join(output)

In [12]:
text = "the cat sat on the mat the cat ate the rat"
model = NGramModel(n=2)
model.train(text)
generated_text = model.generate(start_words=["the"], length=10)
print(generated_text)

the cat ate the mat the mat the cat ate the
